# BCP Appetence Model -- Complete ML Pipeline
**Banque Centrale Populaire -- Data Lakehouse PFE**

**Products:** MaRetraite | Avenir MesEnfants | Epargne Evolution

**Pipeline:**
1. Setup
2. EDA
3. Preprocessing
4. Feature Selection
5. BorderlineSMOTE
6. Model Training with RFE
7. Benchmark
8. Lift & Centile Analysis
9. Ensemble
10. Scoring
11. SHAP

**Reports:** All outputs saved to `/home/jovyan/work/reports/`

---
## 0. Setup

In [ ]:
# 0.1 Set JAVA_HOME -- required for PySpark
import os, subprocess

result = subprocess.run(
    ['find', '/usr', '-name', 'java', '-type', 'f'],
    capture_output=True, text=True
)
java_paths = [p for p in result.stdout.strip().split('\n') if p and 'bin/java' in p]
if java_paths:
    java_home = java_paths[0].replace('/bin/java', '')
    os.environ['JAVA_HOME'] = java_home
    os.environ['PATH'] = java_home + '/bin:' + os.environ['PATH']
    print(f'JAVA_HOME: {java_home}')
else:
    print('ERROR: Java not found')

In [ ]:
# 0.2 Install Python packages
subprocess.run([
    'pip', 'install', '-q',
    'pyspark==3.5.3', 'pandas', 'numpy', 'matplotlib',
    'scikit-learn', 'lightgbm', 'xgboost', 'imbalanced-learn',
    'shap', 'mlflow'
], check=True)
print('All packages installed')

In [ ]:
# 0.3 Start Spark session
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName('BCP_Appetence') \
    .master('spark://spark-master:7077') \
    .config('spark.hadoop.fs.defaultFS', 'hdfs://namenode:9000') \
    .config('spark.driver.memory', '2g') \
    .config('spark.executor.memory', '2g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark ready')
print(spark)

In [ ]:
# 0.4 Global config
import pandas as pd
import numpy as np
import gc, warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# HDFS paths
GOLD_PATH  = 'hdfs://namenode:9000/warehouse/gold/master_table'

# Column groups
LABELS     = ['label_maRetraite', 'label_avenirMesEnfants', 'label_epargneEvolution']
DROP_COLS  = ['RADICAL', 'first_account_date', 'LIBELLE_VILLE', 'TAILLE_ENTREPRI', 'has_valid_carte']
CAT_COLS   = ['GENDER', 'MARITAL_STATUS', 'CUSTOMER_RATING', 'CODE_VILLE', 'BPR']

# Primary evaluation metric: Lift@40%
# 40% of 3.2M clients = ~1.3M -- realistic outreach campaign size
# Lift@40% = 2.0 means 2x more subscribers found vs random contact
LIFT_K     = 0.40

# Reports folder -- all CSVs and plots saved here for the final report
REPORT_DIR = '/home/jovyan/work/reports'
os.makedirs(REPORT_DIR, exist_ok=True)
print(f'Reports directory: {REPORT_DIR}')

# Load Gold master table
df = spark.read.parquet(GOLD_PATH)
print(f'Rows   : {df.count():,}')
print(f'Columns: {len(df.columns)}')

---
## 1. EDA

Exploratory analysis on the full 3.2M client Gold table using pure Spark.

In [ ]:
# 1.1 Label distribution
# All 3 products are heavily imbalanced.
# A naive model predicting 0 for everyone gets 96-99.6% accuracy -- but zero business value.
ld = df.agg(
    F.count('RADICAL').alias('total_clients'),
    F.sum('label_maRetraite').alias('maRetraite'),
    F.sum('label_avenirMesEnfants').alias('avenir'),
    F.sum('label_epargneEvolution').alias('epargne'),
).toPandas()
total = ld['total_clients'].iloc[0]
for c in ['maRetraite', 'avenir', 'epargne']:
    ld[f'{c}_positive_rate'] = (ld[c] / total).round(4)
ld.to_csv(f'{REPORT_DIR}/01_label_distribution.csv', index=False)
print(ld.to_string())
print('\nSaved: 01_label_distribution.csv')

In [ ]:
# 1.2 Missing values per column
# Most columns should be 0 after Silver fillna(0).
# Only demographic columns are expected to have nulls.
n_rows = df.count()
rows = []
for c in df.columns:
    n = df.filter(F.col(c).isNull()).count()
    if n > 0:
        rows.append({'column': c, 'null_count': n, 'null_pct': round(n / n_rows * 100, 2)})
mdf = pd.DataFrame(rows).sort_values('null_count', ascending=False)
mdf.to_csv(f'{REPORT_DIR}/02_missing_values.csv', index=False)
print(mdf.to_string())
print('\nSaved: 02_missing_values.csv')

In [ ]:
# 1.3 Outlier check -- percentiles on financial columns
# Financial columns contain extreme values from corporate accounts.
# We check p50/p75/p95/p99/max to determine cap values.
fcols = [
    'avg_balance', 'max_balance', 'total_flux_cred', 'total_gab_amount',
    'total_tpe_amount', 'total_retrait_amount', 'total_virement_amount',
    'savings_ratio', 'avg_monthly_spend'
]
rows = []
for c in fcols:
    p = df.select(
        F.percentile_approx(c, [0.5, 0.75, 0.95, 0.99, 1.0]).alias('p')
    ).collect()[0]['p']
    rows.append({
        'column': c, 'p50': p[0], 'p75': p[1], 'p95': p[2],
        'p99': p[3], 'max': p[4],
        'max_to_p99_ratio': round(p[4] / p[3], 1) if p[3] and p[3] > 0 else None
    })
odf = pd.DataFrame(rows)
odf.to_csv(f'{REPORT_DIR}/03_outlier_percentiles.csv', index=False)
print(odf.to_string())
print('\nSaved: 03_outlier_percentiles.csv')

In [ ]:
# 1.4 Age distribution by label
# Age differs between subscribers and non-subscribers,
# especially for Epargne Evolution (retirement-age clients avg 51.3 vs 44.8).
age_rows = []
for label in LABELS:
    r = df.groupBy(label).agg(
        F.avg('age').alias('avg_age'),
        F.min('age').alias('min_age'),
        F.max('age').alias('max_age'),
        F.count('*').alias('n')
    ).toPandas()
    r['product'] = label
    age_rows.append(r)
    print(f'\n=== {label} ===')
    print(r.to_string())
pd.concat(age_rows).to_csv(f'{REPORT_DIR}/04_age_by_label.csv', index=False)
print('\nSaved: 04_age_by_label.csv')

In [ ]:
# 1.5 Correlation of numeric features with each label
# nb_insurance_products is expected to dominate -- clients with Attamine/Injad
# are far more likely to subscribe to savings products (corr=0.43 for maRetraite).
ncols = [
    'age', 'avg_balance', 'max_balance', 'total_flux_cred',
    'nb_gab_transactions', 'nb_tpe_transactions', 'nb_online_transactions',
    'has_digital_product', 'has_carte', 'has_pack', 'has_vignette',
    'savings_ratio', 'digital_score', 'spending_diversity',
    'product_breadth', 'balance_trend', 'avg_monthly_spend',
    'anciennete_days', 'nb_accounts', 'nb_insurance_products'
]
rows = []
for label in LABELS:
    for col in ncols:
        try:
            rows.append({'label': label, 'feature': col,
                         'correlation': round(df.stat.corr(col, label), 4)})
        except:
            pass
cdf = pd.DataFrame(rows)
cdf.to_csv(f'{REPORT_DIR}/05_correlations.csv', index=False)
for label in LABELS:
    print(f'\n=== Correlation with {label} ===')
    s = cdf[cdf['label'] == label].sort_values('correlation', key=abs, ascending=False)
    print(s[['feature', 'correlation']].to_string(index=False))
print('\nSaved: 05_correlations.csv')

---
## 2. Preprocessing

Outlier capping at 99th percentile, categorical encoding, and stratified 80/20 split.
All decisions saved to reports.

In [ ]:
# 2.1 Cap outliers at 99th percentile
# Corporate accounts inflate financial columns by 10x-28000x vs the 99th percentile.
# We cap at p99 to remove their distorting effect while preserving retail client signal.
cap_cols = [
    'avg_balance', 'max_balance', 'min_balance', 'last_balance',
    'total_flux_cred', 'avg_flux_cred', 'max_flux_cred',
    'total_gab_amount', 'avg_gab_amount', 'max_gab_amount',
    'total_tpe_amount', 'avg_tpe_amount', 'max_tpe_amount',
    'total_retrait_amount', 'avg_retrait_amount',
    'total_online_amount', 'avg_online_amount',
    'total_payfac_amount', 'total_virement_amount',
    'total_depot_amount', 'total_mad_amount',
    'savings_ratio', 'avg_monthly_spend', 'balance_trend'
]
# Compute 99th percentile for all columns in one Spark pass
pcts = df.select(
    [F.percentile_approx(c, 0.99).alias(c) for c in cap_cols]
).collect()[0].asDict()

df_clean = df
rep = []
for col in cap_cols:
    v = pcts[col]
    if v and v > 0:
        df_clean = df_clean.withColumn(col, F.least(F.col(col), F.lit(v)))
        rep.append({'column': col, 'cap_p99': v})
# NOMBRE_ENFANT: raw max was 99 -- clearly a data entry error, cap at 10
df_clean = df_clean.withColumn('NOMBRE_ENFANT', F.least(F.col('NOMBRE_ENFANT'), F.lit(10)))
rep.append({'column': 'NOMBRE_ENFANT', 'cap_p99': 10})

pd.DataFrame(rep).to_csv(f'{REPORT_DIR}/06_outlier_caps.csv', index=False)
print(pd.DataFrame(rep).to_string())
print(f'\nCapped {len(rep)} columns')
print('Saved: 06_outlier_caps.csv')

In [ ]:
# 2.2 Encode categorical columns + drop irrelevant columns
# StringIndexer with handleInvalid='keep' handles nulls and unseen values gracefully.
# Dropped columns and reasons:
#   RADICAL          -- client identifier, not a feature
#   first_account_date -- anciennete_days already derived from this
#   LIBELLE_VILLE    -- free text, 135k nulls, CODE_VILLE is cleaner
#   TAILLE_ENTREPRI  -- 99.97% null
#   has_valid_carte  -- all zeros (all cards have expired validity dates in Silver)
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

indexers = [
    StringIndexer(inputCol=c, outputCol=f'{c}_idx', handleInvalid='keep')
    for c in CAT_COLS
]
df_enc = Pipeline(stages=indexers).fit(df_clean).transform(df_clean)
df_enc = df_enc.drop(*(CAT_COLS + DROP_COLS)).fillna(0)

# Save preprocessing decisions
pd.DataFrame(
    [{'step': 'Dropped', 'column': c, 'reason': 'identifier or low value'} for c in DROP_COLS] +
    [{'step': 'StringIndexer', 'column': c, 'reason': 'categorical encoding'} for c in CAT_COLS]
).to_csv(f'{REPORT_DIR}/07_preprocessing_decisions.csv', index=False)

print(f'Columns after encoding: {len(df_enc.columns)}')
print('Saved: 07_preprocessing_decisions.csv')

In [ ]:
# 2.3 Save preprocessed table to HDFS for reuse
df_enc.write.mode('overwrite').parquet(
    'hdfs://namenode:9000/warehouse/gold/master_preprocessed'
)
print('Saved to HDFS: /warehouse/gold/master_preprocessed')

In [ ]:
# 2.4 Load into Pandas from local disk
# master_preprocessed folder must be in /home/jovyan/work/
pdf = pd.read_parquet('/home/jovyan/work/master_preprocessed')
print(f'Shape : {pdf.shape}')
print(f'Memory: {pdf.memory_usage(deep=True).sum() / 1024**3:.2f} GB')
pdf.head()

In [ ]:
# 2.5 Stratified 80/20 train/validation split
# Stratified on label_maRetraite to preserve the 3.95% positive rate in both splits.
from sklearn.model_selection import train_test_split

FEATURE_COLS = [c for c in pdf.columns if c not in LABELS]
print(f'Feature columns: {len(FEATURE_COLS)}')

X     = pdf[FEATURE_COLS].fillna(0)
y_mar = pdf['label_maRetraite'].copy()
y_ave = pdf['label_avenirMesEnfants'].copy()
y_epe = pdf['label_epargneEvolution'].copy()

# Free original dataframe immediately -- saves ~1.7GB RAM
del pdf
gc.collect()
print('pdf freed from memory')

(X_train, X_val,
 y_train_mar, y_val_mar,
 y_train_ave, y_val_ave,
 y_train_epe, y_val_epe) = train_test_split(
    X, y_mar, y_ave, y_epe,
    test_size=0.2, stratify=y_mar, random_state=42
)

n_pos = int(y_train_mar.sum())
n_neg = len(y_train_mar) - n_pos

split_df = pd.DataFrame([
    {'split': 'train', 'n': len(X_train),
     'maRetraite_rate': round(y_train_mar.mean(), 4),
     'avenir_rate': round(y_train_ave.mean(), 4),
     'epargne_rate': round(y_train_epe.mean(), 4)},
    {'split': 'val', 'n': len(X_val),
     'maRetraite_rate': round(y_val_mar.mean(), 4),
     'avenir_rate': round(y_val_ave.mean(), 4),
     'epargne_rate': round(y_val_epe.mean(), 4)},
])
split_df.to_csv(f'{REPORT_DIR}/08_train_val_split.csv', index=False)
print(split_df.to_string())
print(f'\nn_pos={n_pos:,}  n_neg={n_neg:,}  scale_pos_weight={n_neg/n_pos:.1f}x')
print('Saved: 08_train_val_split.csv')

---
## 3. Feature Selection

**Step 1:** Train a quick 100-tree LightGBM to rank all features by importance.

**Step 2:** Drop features with zero importance -- they contribute nothing to the model.

**Step 3:** Drop one feature from each highly correlated pair (correlation > 0.9),
keeping the one with higher importance.

All dropped features and reasons are saved to the reports folder.

In [ ]:
# 3.1 Quick LightGBM to rank features
# Sample 200k rows -- enough for reliable importance rankings without crashing
import lightgbm as lgb

X_train_sample = X_train.sample(n=200_000, random_state=42)
y_train_sample = y_train_mar.loc[X_train_sample.index]

print(f'Sample shape   : {X_train_sample.shape}')
print(f'Sample pos rate: {y_train_sample.mean():.4f}')

n_pos_s = int(y_train_sample.sum())
n_neg_s = len(y_train_sample) - n_pos_s

quick_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=31,
    scale_pos_weight=n_neg_s / n_pos_s,
    objective='binary',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
quick_model.fit(X_train_sample, y_train_sample)
del X_train_sample, y_train_sample
import gc; gc.collect()
print('Quick model trained')

In [ ]:
# 3.2 Extract and save feature importances
imp = pd.Series(
    quick_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

idf = imp.reset_index()
idf.columns = ['feature', 'importance']
idf['status'] = idf['importance'].apply(lambda x: 'zero_importance' if x == 0 else 'kept')
idf.to_csv(f'{REPORT_DIR}/09_feature_importances.csv', index=False)

print(f'Total features      : {len(imp)}')
print(f'Zero importance     : {(imp == 0).sum()}')
print(f'Non-zero importance : {(imp > 0).sum()}')
print('\n=== Top 30 features ===')
print(imp.head(30))
print('\n=== DROPPED -- Zero importance ===')
print(imp[imp == 0].index.tolist())
print('\nSaved: 09_feature_importances.csv')

In [ ]:
# 3.3 Drop correlated pairs -- keep the more important feature
# Use a sample for correlation computation -- 200k rows is sufficient
useful = imp[imp > 0].index.tolist()
print(f'Features after dropping zeros: {len(useful)}')

# Sample for correlation computation to avoid memory crash
X_corr_sample = X_train[useful].sample(n=200_000, random_state=42)
corr_matrix = X_corr_sample.corr().abs()
del X_corr_sample
gc.collect()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

corr_pairs = []
drop_set = set()
for col in upper.columns:
    for other in upper[col][upper[col] > 0.9].index.tolist():
        if imp[col] >= imp[other]:
            drop_set.add(other)
            corr_pairs.append({
                'kept': col, 'dropped': other,
                'correlation': round(upper[col][other], 4),
                'importance_kept': int(imp[col]),
                'importance_dropped': int(imp[other])
            })
        else:
            drop_set.add(col)
            corr_pairs.append({
                'kept': other, 'dropped': col,
                'correlation': round(upper[col][other], 4),
                'importance_kept': int(imp[other]),
                'importance_dropped': int(imp[col])
            })

corr_df = pd.DataFrame(corr_pairs)
corr_df.to_csv(f'{REPORT_DIR}/10_dropped_correlated_features.csv', index=False)
print('\n=== DROPPED -- Highly correlated (> 0.9) ===')
print(corr_df.to_string())

final_features = [f for f in useful if f not in drop_set]
pd.DataFrame({
    'feature': final_features,
    'importance': [int(imp[f]) for f in final_features]
}).sort_values('importance', ascending=False).to_csv(
    f'{REPORT_DIR}/11_final_feature_list.csv', index=False
)
print(f'\nFinal feature count: {len(final_features)}')
print(final_features)
print('\nSaved: 10_dropped_correlated_features.csv  11_final_feature_list.csv')

In [ ]:
# 3.4 Apply feature selection and free memory
X_train_fs = X_train[final_features].copy()
X_val_fs   = X_val[final_features].copy()

# Free the full feature matrices -- saves RAM before SMOTE
del X_train, X_val
gc.collect()

print(f'X_train_fs: {X_train_fs.shape}')
print(f'X_val_fs  : {X_val_fs.shape}')

---
## 4. BorderlineSMOTE Resampling

### Why Lift@40% as primary metric?

We evaluate all models using **Lift@40%** -- the lift achieved when contacting the top 40% of scored clients.

- The commercial team cannot contact all 3.2M clients
- 40% (~1.3M clients) is a realistic large-scale outreach campaign
- **Lift@40% = 2.0** means the top 40% of scored clients contains twice as many subscribers as random contact
- This directly translates to campaign cost savings
- Standard evaluation metric in French retail banking CRM scoring

### Why BorderlineSMOTE over standard SMOTE?

Standard SMOTE generates synthetic minority samples uniformly across the minority space.
**BorderlineSMOTE** focuses only on samples near the class boundary -- subscribers who look like non-subscribers.
These are the hardest cases and the most informative for the model to learn from.

The `kind='borderline-1'` setting synthesizes only from minority samples that have majority neighbors.

### Target: 200k rows, 10% positive rate
- 20,000 positive examples | 180,000 negative examples
- Closer to reality than 50/50 -- avoids overconfident predictions

In [ ]:
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

TARGET_POS   = 20_000    # synthetic + real positive examples
TARGET_NEG   = 180_000   # undersampled negative examples
TARGET_TOTAL = TARGET_POS + TARGET_NEG  # 200k total

print(f'Target positives: {TARGET_POS:,}')
print(f'Target negatives: {TARGET_NEG:,}')
print(f'Target total    : {TARGET_TOTAL:,}')
print(f'Target pos rate : {TARGET_POS/TARGET_TOTAL:.0%}')
print(f'\nOriginal train: n_pos={n_pos:,}  n_neg={n_neg:,}  rate={n_pos/len(y_train_mar):.4f}')

# Step 1: BorderlineSMOTE oversample positives toward TARGET_POS
smote = BorderlineSMOTE(
    sampling_strategy=TARGET_POS / n_neg,
    random_state=42,
    k_neighbors=5,
    m_neighbors=10,
    kind='borderline-1'
)
# Step 2: RandomUnderSampler bring negatives down to TARGET_NEG
under = RandomUnderSampler(
    sampling_strategy=TARGET_POS / TARGET_NEG,
    random_state=42
)

pipeline_smote = ImbPipeline([('smote', smote), ('under', under)])

print('\nApplying BorderlineSMOTE + undersampling...')
X_res, y_res = pipeline_smote.fit_resample(X_train_fs, y_train_mar)

samp_df = pd.DataFrame([
    {'step': 'original_train', 'n': len(y_train_mar),
     'n_pos': int(y_train_mar.sum()), 'pos_rate': round(y_train_mar.mean(), 4)},
    {'step': 'after_borderline_smote', 'n': len(y_res),
     'n_pos': int(y_res.sum()), 'pos_rate': round(y_res.mean(), 4)},
])
samp_df.to_csv(f'{REPORT_DIR}/12_sampling_report.csv', index=False)
print(samp_df.to_string())
print('\nSaved: 12_sampling_report.csv')

---
## 5. Evaluation Function -- Lift@40%

Lift@40% formula:
1. Sort all validation clients by predicted score descending
2. Take the top 40%
3. Lift = (% of real subscribers in top 40%) / 40%

Lift=2.0 means the top 40% contains 2x more subscribers than random.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
import mlflow
import mlflow.lightgbm
import mlflow.xgboost
import mlflow.sklearn

mlflow.set_tracking_uri('http://mlflow:5000')
mlflow.set_experiment('bcp_appetence_maRetraite')

def lift_at_k(y_true, y_proba, k=0.40):
    arr = np.array(y_true)
    idx = np.argsort(y_proba)[::-1]
    n_top = int(len(arr) * k)
    cap = arr[idx[:n_top]].sum() / arr.sum()
    lift = cap / k
    return round(float(lift), 4), round(float(cap), 4)

def evaluate(name, y_true, y_proba, k=LIFT_K):
    lift, cap = lift_at_k(y_true, y_proba, k)
    yp = (y_proba >= 0.5).astype(int)
    return {
        'model':         name,
        'lift_40pct':    lift,    # PRIMARY METRIC
        'capture_40pct': cap,    # % of subscribers captured in top 40%
        'pr_auc':        round(float(average_precision_score(y_true, y_proba)), 4),
        'roc_auc':       round(float(roc_auc_score(y_true, y_proba)), 4),
        'f1':            round(float(f1_score(y_true, yp, zero_division=0)), 4),
    }

results = []
print('Evaluation function ready')
print(f'Primary metric: Lift@{int(LIFT_K*100)}%')
print('Interpretation: Lift=2.0 means top 40% of clients captures 2x more subscribers than random')

---
## 6. Model Training with RFE

**Recursive Feature Elimination (RFE)** on the 200k BorderlineSMOTE dataset.

For each model:
1. Train on current feature set, evaluate Lift@40% on original validation set
2. Drop the bottom 10% of features by importance
3. Repeat until fewer than 10 features remain
4. Keep the feature set that gave the best Lift@40%
5. Retrain final model with more trees on the optimal feature subset

Each model gets its own optimal feature subset. RFE history saved per model.

In [ ]:
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

def get_importances(model, X, y, model_type):
    # Train model and return feature importances
    if model_type == 'lgbm':
        model.fit(X, y, eval_set=[(X, y)],
                  callbacks=[lgb.early_stopping(20), lgb.log_evaluation(-1)])
    elif model_type == 'xgb':
        model.fit(X, y, eval_set=[(X, y)], verbose=False)
    else:
        model.fit(X, y)
    return model.feature_importances_

def rfe_run(factory, model_type, X_smote, y_smote, X_val, y_val,
            name, step=0.1, min_features=10):
    # Run RFE on SMOTE training set, evaluate on original validation set.
    feats = list(X_smote.columns)
    best_lift = -1
    best_feats = feats[:]
    history = []
    rnd = 0
    print(f'\nRFE -- {name} | start={len(feats)} features | step={int(step*100)}%')
    print('-' * 60)
    while len(feats) >= min_features:
        rnd += 1
        model = factory()
        imps = get_importances(model, X_smote[feats], y_smote, model_type)
        # Evaluate on ORIGINAL validation set (not SMOTE)
        yp = model.predict_proba(X_val[feats])[:, 1]
        lift, cap = lift_at_k(y_val, yp)
        pr = round(float(average_precision_score(y_val, yp)), 4)
        history.append({
            'round': rnd, 'n_features': len(feats),
            'lift_40pct': lift, 'capture_40pct': cap, 'pr_auc': pr
        })
        print(f'  Round {rnd:>2} | features={len(feats):>3} | lift={lift:.4f} | cap={cap:.4f} | pr={pr:.4f}')
        if lift > best_lift:
            best_lift = lift
            best_feats = feats[:]
        # Drop bottom step% of features by importance
        imp_series = pd.Series(imps, index=feats)
        n_drop = max(1, int(len(feats) * step))
        to_drop = imp_series.nsmallest(n_drop).index.tolist()
        feats = [f for f in feats if f not in to_drop]
        if len(feats) < min_features:
            break
    print(f'\nBest: {len(best_feats)} features | Lift@40%={best_lift:.4f}')
    return best_feats, pd.DataFrame(history)

rfe_results = {}
print('RFE pipeline ready')

In [ ]:
# 6.1 LightGBM RFE
def lgbm_factory():
    return lgb.LGBMClassifier(
        n_estimators=300,    # fewer trees for RFE rounds (speed)
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

with mlflow.start_run(run_name='LightGBM_RFE'):
    # RFE phase -- find optimal feature subset
    bf, hist = rfe_run(lgbm_factory, 'lgbm', X_res, y_res, X_val_fs, y_val_mar, 'LightGBM')

    # Final training -- more trees on the optimal subset
    fm = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    fm.fit(
        X_res[bf], y_res,
        eval_set=[(X_val_fs[bf], y_val_mar)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )
    yp = fm.predict_proba(X_val_fs[bf])[:, 1]
    m = evaluate('LightGBM', y_val_mar, yp)
    m['n_features_rfe'] = len(bf)
    m['best_iteration'] = fm.best_iteration_
    mlflow.log_metrics({k: v for k, v in m.items() if k not in ['model', 'n_features_rfe', 'best_iteration']})
    mlflow.log_params({'n_features_rfe': len(bf), 'sampling': 'BorderlineSMOTE_10pct'})
    mlflow.lightgbm.log_model(fm, artifact_path='lgbm')
    hist['model'] = 'LightGBM'
    hist.to_csv(f'{REPORT_DIR}/13a_rfe_history_LightGBM.csv', index=False)
    rfe_results['LightGBM'] = {'f': bf, 'model': fm, 'h': hist, 'p': yp, 'm': m}
    results.append(m)
    print(f'\nFinal metrics: {m}')
    print('Saved: 13a_rfe_history_LightGBM.csv')

In [ ]:
# 6.2 XGBoost RFE
def xgb_factory():
    return xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='aucpr',
        early_stopping_rounds=20,
        random_state=42,
        verbosity=0,
        n_jobs=-1
    )

with mlflow.start_run(run_name='XGBoost_RFE'):
    bf, hist = rfe_run(xgb_factory, 'xgb', X_res, y_res, X_val_fs, y_val_mar, 'XGBoost')
    fm = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='aucpr',
        early_stopping_rounds=50,
        random_state=42,
        verbosity=0,
        n_jobs=-1
    )
    fm.fit(X_res[bf], y_res, eval_set=[(X_val_fs[bf], y_val_mar)], verbose=False)
    yp = fm.predict_proba(X_val_fs[bf])[:, 1]
    m = evaluate('XGBoost', y_val_mar, yp)
    m['n_features_rfe'] = len(bf)
    mlflow.log_metrics({k: v for k, v in m.items() if k not in ['model', 'n_features_rfe']})
    mlflow.log_params({'n_features_rfe': len(bf), 'sampling': 'BorderlineSMOTE_10pct'})
    mlflow.xgboost.log_model(fm, artifact_path='xgb')
    hist['model'] = 'XGBoost'
    hist.to_csv(f'{REPORT_DIR}/13b_rfe_history_XGBoost.csv', index=False)
    rfe_results['XGBoost'] = {'f': bf, 'model': fm, 'h': hist, 'p': yp, 'm': m}
    results.append(m)
    print(f'\nFinal metrics: {m}')
    print('Saved: 13b_rfe_history_XGBoost.csv')

In [ ]:
# 6.3 Random Forest RFE
def rf_factory():
    return RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=50,
        n_jobs=-1,
        random_state=42
    )

with mlflow.start_run(run_name='RandomForest_RFE'):
    bf, hist = rfe_run(rf_factory, 'rf', X_res, y_res, X_val_fs, y_val_mar, 'RandomForest')
    fm = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=50,
        n_jobs=-1,
        random_state=42
    )
    fm.fit(X_res[bf], y_res)
    yp = fm.predict_proba(X_val_fs[bf])[:, 1]
    m = evaluate('RandomForest', y_val_mar, yp)
    m['n_features_rfe'] = len(bf)
    mlflow.log_metrics({k: v for k, v in m.items() if k not in ['model', 'n_features_rfe']})
    mlflow.log_params({'n_features_rfe': len(bf), 'sampling': 'BorderlineSMOTE_10pct'})
    mlflow.sklearn.log_model(fm, artifact_path='rf')
    hist['model'] = 'RandomForest'
    hist.to_csv(f'{REPORT_DIR}/13c_rfe_history_RandomForest.csv', index=False)
    rfe_results['RandomForest'] = {'f': bf, 'model': fm, 'h': hist, 'p': yp, 'm': m}
    results.append(m)
    print(f'\nFinal metrics: {m}')
    print('Saved: 13c_rfe_history_RandomForest.csv')

In [ ]:
# 6.4 AdaBoost RFE
def ada_factory():
    return AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=3),
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    )

with mlflow.start_run(run_name='AdaBoost_RFE'):
    bf, hist = rfe_run(ada_factory, 'ada', X_res, y_res, X_val_fs, y_val_mar, 'AdaBoost')
    fm = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=3),
        n_estimators=200,
        learning_rate=0.1,
        random_state=42
    )
    fm.fit(X_res[bf], y_res)
    yp = fm.predict_proba(X_val_fs[bf])[:, 1]
    m = evaluate('AdaBoost', y_val_mar, yp)
    m['n_features_rfe'] = len(bf)
    mlflow.log_metrics({k: v for k, v in m.items() if k not in ['model', 'n_features_rfe']})
    mlflow.log_params({'n_features_rfe': len(bf), 'sampling': 'BorderlineSMOTE_10pct'})
    mlflow.sklearn.log_model(fm, artifact_path='ada')
    hist['model'] = 'AdaBoost'
    hist.to_csv(f'{REPORT_DIR}/13d_rfe_history_AdaBoost.csv', index=False)
    rfe_results['AdaBoost'] = {'f': bf, 'model': fm, 'h': hist, 'p': yp, 'm': m}
    results.append(m)
    print(f'\nFinal metrics: {m}')
    print('Saved: 13d_rfe_history_AdaBoost.csv')

In [ ]:
# 6.5 RFE summary + convergence plots
rfe_sum = pd.DataFrame([{
    'model': n,
    'features_before': len(final_features),
    'features_after_rfe': len(r['f']),
    'features_dropped': len(final_features) - len(r['f']),
    'lift_40pct': r['m']['lift_40pct'],
    'pr_auc': r['m']['pr_auc']
} for n, r in rfe_results.items()])
rfe_sum.to_csv(f'{REPORT_DIR}/14_rfe_summary.csv', index=False)

# Feature sets per model
pd.DataFrame({
    n: pd.Series(sorted(r['f'])) for n, r in rfe_results.items()
}).to_csv(f'{REPORT_DIR}/15_rfe_features_per_model.csv', index=False)

# Consensus: features selected by ALL 4 models
consensus = sorted(set.intersection(*[set(r['f']) for r in rfe_results.values()]))
pd.DataFrame({'feature': consensus}).to_csv(f'{REPORT_DIR}/16_rfe_consensus_features.csv', index=False)

print('=== RFE Summary ===')
print(rfe_sum.to_string())
print(f'\nConsensus features selected by all 4 models ({len(consensus)}):')
print(consensus)

# Convergence plots
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
colors = ['steelblue', 'darkorange', 'green', 'red']
for i, (name, res) in enumerate(rfe_results.items()):
    h = res['h'].sort_values('n_features')
    br = h.loc[h['lift_40pct'].idxmax()]
    axes[i].plot(h['n_features'], h['lift_40pct'], color=colors[i], lw=2, marker='o', ms=5)
    axes[i].axvline(x=br['n_features'], color='black', linestyle='--', alpha=0.7,
                    label=f'Best: {int(br["n_features"])} features')
    axes[i].scatter([br['n_features']], [br['lift_40pct']], color='black', zorder=5, s=80)
    axes[i].set_title(f'{name} -- RFE Convergence', fontweight='bold')
    axes[i].set_xlabel('Number of features')
    axes[i].set_ylabel('Lift@40%')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
    axes[i].invert_xaxis()  # left = more features, right = fewer
plt.suptitle('RFE Convergence -- Lift@40% vs Number of Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/17_rfe_convergence_curves.png', dpi=150)
plt.show()
print('Saved: 14-17 RFE reports')

# Assign model variables for downstream sections
lgbm = rfe_results['LightGBM']['model']
xgbm = rfe_results['XGBoost']['model']
rf   = rfe_results['RandomForest']['model']
ada  = rfe_results['AdaBoost']['model']
y_proba_lgbm = rfe_results['LightGBM']['p']
y_proba_xgb  = rfe_results['XGBoost']['p']
y_proba_rf   = rfe_results['RandomForest']['p']
y_proba_ada  = rfe_results['AdaBoost']['p']
lgbm_feats = rfe_results['LightGBM']['f']
xgb_feats  = rfe_results['XGBoost']['f']
rf_feats   = rfe_results['RandomForest']['f']
ada_feats  = rfe_results['AdaBoost']['f']
print('\nAll model variables assigned for downstream use')

---
## 7. Benchmark

All 4 models ranked by primary metric Lift@40%.

In [ ]:
# Benchmark -- sort by Lift@40% (primary metric)
rdf = pd.DataFrame(results).sort_values('lift_40pct', ascending=False)
rdf.index = range(1, len(rdf) + 1)
rdf.to_csv(f'{REPORT_DIR}/18_benchmark_individual_models.csv', index=False)

print(' Model Benchmark -- sorted by Lift@40% ')
print(rdf.to_string())
print('\nSaved: 18_benchmark_individual_models.csv')

best_model_name = rdf.iloc[0]['model']
print(f'\nBest model: {best_model_name}')

---
## 8. Lift & Centile Analysis

Decile lift table, top centile analysis (1/2/5/10/20/30/40/50%), and lift curves for all 4 models.

In [ ]:
# Model probabilities dictionary for easy iteration
model_probas = {
    'LightGBM':     y_proba_lgbm,
    'XGBoost':      y_proba_xgb,
    'RandomForest': y_proba_rf,
    'AdaBoost':     y_proba_ada,
}

def lift_table(y_true, y_proba, n_bins=10, label='model'):
    # Compute lift table by decile
    dfl = pd.DataFrame({'y_true': np.array(y_true), 'y_proba': y_proba})
    dfl = dfl.sort_values('y_proba', ascending=False).reset_index(drop=True)
    dfl['decile'] = pd.qcut(dfl.index, n_bins, labels=False) + 1
    overall_rate = np.array(y_true).mean()
    t = dfl.groupby('decile').agg(
        n=('y_true', 'count'), n_pos=('y_true', 'sum'), avg_score=('y_proba', 'mean')
    ).reset_index()
    t['positive_rate'] = (t['n_pos'] / t['n']).round(4)
    t['lift'] = (t['positive_rate'] / overall_rate).round(4)
    t['cumulative_pct_captured'] = (t['n_pos'].cumsum() / t['n_pos'].sum() * 100).round(2)
    t['model'] = label
    print(f'\n=== Lift Table -- {label} (overall rate={overall_rate:.4f}) ===')
    print(t.to_string(index=False))
    return t

def centile_analysis(y_true, y_proba, centiles=[1, 2, 5, 10, 20, 30, 40, 50], label='model'):
    # Top centile analysis: precision, capture rate, and lift at each centile
    arr = np.array(y_true)
    idx = np.argsort(y_proba)[::-1]
    total_pos = arr.sum()
    overall_rate = arr.mean()
    n = len(arr)
    rows = []
    print(f'\n=== Top Centile Analysis -- {label} ===')
    print(f'{"Centile":<10} {"N clients":<12} {"N positives":<14} {"Capture %":<12} {"Precision":<12} {"Lift":<8}')
    print('-' * 70)
    for c in centiles:
        nt = int(n * c / 100)
        np_c = arr[idx[:nt]].sum()
        cap = np_c / total_pos * 100
        prec = np_c / nt if nt > 0 else 0
        lift = prec / overall_rate if overall_rate > 0 else 0
        marker = ' <-- PRIMARY METRIC' if c == 40 else ''
        print(f'Top {c}%{"":<5} {nt:<12,} {int(np_c):<14,} {cap:<12.1f} {prec:<12.4f} {lift:<8.2f}x{marker}')
        rows.append({
            'model': label, 'centile': c, 'n_clients': nt,
            'n_positives': int(np_c), 'capture_pct': round(cap, 2),
            'precision': round(float(prec), 4), 'lift': round(float(lift), 4)
        })
    return pd.DataFrame(rows)

# Run for all 4 models
all_lt = []
all_ct = []
for name, proba in model_probas.items():
    all_lt.append(lift_table(y_val_mar, proba, label=name))
    all_ct.append(centile_analysis(y_val_mar, proba, label=name))

pd.concat(all_lt).to_csv(f'{REPORT_DIR}/19_lift_tables_all_models.csv', index=False)
pd.concat(all_ct).to_csv(f'{REPORT_DIR}/20_centile_analysis_all_models.csv', index=False)
print('\nSaved: 19_lift_tables_all_models.csv')
print('Saved: 20_centile_analysis_all_models.csv')

In [ ]:
# Lift curves -- cumulative gains and lift for all 4 models on same plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['steelblue', 'darkorange', 'green', 'red']

for (name, proba), color in zip(model_probas.items(), colors):
    arr = np.array(y_val_mar)
    idx = np.argsort(proba)[::-1]
    total_pos = arr.sum()
    n = len(arr)
    step = max(1, n // 200)  # sample 200 points for smooth curve
    pp, pc, lv = [], [], []
    for i in range(1, n + 1, step):
        p = i / n * 100
        c = arr[idx[:i]].sum() / total_pos * 100
        pp.append(p); pc.append(c); lv.append(c / p if p > 0 else 1)
    axes[0].plot(pp, pc, color=color, lw=2, label=name)
    axes[1].plot(pp, lv, color=color, lw=2, label=name)

# Baselines
axes[0].plot([0, 100], [0, 100], '--', color='gray', label='Random baseline')
axes[1].axhline(y=1, color='gray', linestyle='--', label='No lift (random)')

# Mark the 40% cutoff -- our primary evaluation point
for ax in axes:
    ax.axvline(x=40, color='black', linestyle=':', alpha=0.7, label='40% cutoff (primary)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set(xlabel='% Population contacted', ylabel='% Subscribers captured',
            title='Cumulative Gains Curve')
axes[1].set(xlabel='% Population contacted', ylabel='Lift factor',
            title='Lift Curve')

plt.suptitle('Lift Analysis -- MaRetraite -- All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/21_lift_curves_all_models.png', dpi=150)
plt.show()
print('Saved: 21_lift_curves_all_models.png')

---
## 9. Ensemble

Three ensemble strategies:
1. **Simple average** -- equal weight to all 4 models
2. **Weighted average** -- weight each model by its individual Lift@40%
3. **Stacking** -- LightGBM meta-model trained on the 4 model scores

In [ ]:
# 9.1 Simple average -- all models equal weight
ens_avg = (y_proba_lgbm + y_proba_xgb + y_proba_rf + y_proba_ada) / 4

with mlflow.start_run(run_name='Ensemble_Average'):
    m = evaluate('Ensemble_Average', y_val_mar, ens_avg)
    mlflow.log_metrics({k: v for k, v in m.items() if k != 'model'})
    mlflow.log_param('strategy', 'simple_average')
    results.append(m)
    print(m)

# 9.2 Weighted average -- weight proportional to individual Lift@40%
lift_scores = {n: r['m']['lift_40pct'] for n, r in rfe_results.items()}
total_lift  = sum(lift_scores.values())
weights     = {k: v / total_lift for k, v in lift_scores.items()}
print('\nEnsemble weights (proportional to Lift@40%):')
for k, v in weights.items():
    print(f'  {k:<15} {round(v, 4)}')

ens_w = (
    weights['LightGBM']     * y_proba_lgbm +
    weights['XGBoost']      * y_proba_xgb  +
    weights['RandomForest'] * y_proba_rf   +
    weights['AdaBoost']     * y_proba_ada
)
with mlflow.start_run(run_name='Ensemble_Weighted'):
    m = evaluate('Ensemble_Weighted', y_val_mar, ens_w)
    mlflow.log_metrics({k: v for k, v in m.items() if k != 'model'})
    mlflow.log_param('strategy', 'weighted_by_lift40pct')
    results.append(m)
    print(m)

# 9.3 Stacking -- LightGBM meta-model on top of 4 model scores
# Meta-features: predicted probabilities from each base model on the training set
meta_tr = np.column_stack([
    lgbm.predict_proba(X_train_fs[lgbm_feats])[:, 1],
    xgbm.predict_proba(X_train_fs[xgb_feats])[:, 1],
    rf.predict_proba(X_train_fs[rf_feats])[:, 1],
    ada.predict_proba(X_train_fs[ada_feats])[:, 1],
])
meta_vl = np.column_stack([y_proba_lgbm, y_proba_xgb, y_proba_rf, y_proba_ada])

meta_model = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05, num_leaves=15,
    objective='binary', random_state=42, n_jobs=-1, verbose=-1
)
meta_model.fit(meta_tr, y_train_mar)
ens_s = meta_model.predict_proba(meta_vl)[:, 1]

with mlflow.start_run(run_name='Ensemble_Stacking'):
    m = evaluate('Ensemble_Stacking', y_val_mar, ens_s)
    mlflow.log_metrics({k: v for k, v in m.items() if k != 'model'})
    mlflow.log_param('strategy', 'stacking_lgbm_meta')
    results.append(m)
    print(m)

In [ ]:
# Final benchmark -- all models + all ensembles
fdf = pd.DataFrame(results).sort_values('lift_40pct', ascending=False)
fdf.index = range(1, len(fdf) + 1)
fdf.to_csv(f'{REPORT_DIR}/22_final_benchmark_all_models.csv', index=False)

# Save ensemble weights
pd.DataFrame([{
    'model': k,
    'individual_lift_40pct': lift_scores.get(k),
    'ensemble_weight': round(weights.get(k, 0), 4)
} for k in rfe_results]).to_csv(f'{REPORT_DIR}/23_ensemble_weights.csv', index=False)

print('=== FINAL BENCHMARK -- All models + Ensembles ===')
print(fdf.to_string())
print('\nSaved: 22_final_benchmark_all_models.csv')
print('Saved: 23_ensemble_weights.csv')

---
## 10. Scoring -- All 3.2M Clients

Score the full client base with all 4 models + weighted ensemble.
Output: ranked client list with decile assignment.

In [ ]:
# Load full preprocessed data
pdf_full = pd.read_parquet('/home/jovyan/work/master_preprocessed')
print(f'Full dataset: {len(pdf_full):,} clients')

# Score with each model using its own RFE-selected feature subset
pdf_full['score_lgbm'] = lgbm.predict_proba(pdf_full[lgbm_feats].fillna(0))[:, 1]
pdf_full['score_xgb']  = xgbm.predict_proba(pdf_full[xgb_feats].fillna(0))[:, 1]
pdf_full['score_rf']   = rf.predict_proba(pdf_full[rf_feats].fillna(0))[:, 1]
pdf_full['score_ada']  = ada.predict_proba(pdf_full[ada_feats].fillna(0))[:, 1]

# Weighted ensemble score
pdf_full['score_ensemble'] = (
    weights['LightGBM']     * pdf_full['score_lgbm'] +
    weights['XGBoost']      * pdf_full['score_xgb']  +
    weights['RandomForest'] * pdf_full['score_rf']   +
    weights['AdaBoost']     * pdf_full['score_ada']
)

# Rank clients by ensemble score (rank 1 = highest subscription propensity)
pdf_full['rank']   = pdf_full['score_ensemble'].rank(ascending=False).astype(int)
pdf_full['decile'] = pd.qcut(pdf_full['score_ensemble'], q=10, labels=False) + 1

# Save scored output
score_cols = ['score_lgbm', 'score_xgb', 'score_rf', 'score_ada', 'score_ensemble']
out = pdf_full[LABELS + score_cols + ['rank', 'decile']].sort_values('score_ensemble', ascending=False)
out.to_parquet('/home/jovyan/work/scores_maRetraite.parquet', index=False)

# Score distribution by decile
dec = out.groupby('decile').agg(
    n=('score_ensemble', 'count'),
    avg_score=('score_ensemble', 'mean'),
    n_subscribers=('label_maRetraite', 'sum')
).reset_index()
dec['positive_rate'] = (dec['n_subscribers'] / dec['n']).round(4)
dec.to_csv(f'{REPORT_DIR}/24_score_distribution_by_decile.csv', index=False)

print(f'\nScored {len(pdf_full):,} clients')
print(f'Top 10% to contact: {len(out[out["decile"]==10]):,}')
print(f'Top 40% to contact: {len(out[out["decile"]>=7]):,}')
print('\nScore distribution by decile:')
print(dec.to_string())
print('\nSaved: scores_maRetraite.parquet')
print('Saved: 24_score_distribution_by_decile.csv')

---
## 11. SHAP Explainability

SHAP provides both global feature importance and per-client explanations.
We use the LightGBM model on a 5000-client sample from the validation set.

In [ ]:
import shap

# Sample 5000 clients for SHAP -- full val set would take too long
X_shap = X_val_fs[lgbm_feats].sample(5000, random_state=42)
print(f'SHAP sample: {X_shap.shape}')

# TreeExplainer is fast for tree-based models
explainer   = shap.TreeExplainer(lgbm)
shap_values = explainer.shap_values(X_shap)

# Handle both single-output (array) and multi-output (list) SHAP values
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

# Save mean absolute SHAP values -- global feature importance
shap_df = pd.DataFrame({
    'feature': X_shap.columns,
    'mean_abs_shap': np.abs(sv).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)
shap_df.to_csv(f'{REPORT_DIR}/25_shap_feature_importance.csv', index=False)

print('\nTop 20 features by mean |SHAP|:')
print(shap_df.head(20).to_string())

# Beeswarm plot -- magnitude and direction of each feature's impact
shap.summary_plot(sv, X_shap, max_display=20, show=False)
plt.title('SHAP Feature Importance -- LightGBM -- MaRetraite')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/26_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

# Bar plot -- mean absolute SHAP value per feature
shap.summary_plot(sv, X_shap, plot_type='bar', max_display=20, show=False)
plt.title('SHAP Mean Absolute Impact -- MaRetraite')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/27_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSaved: 25_shap_feature_importance.csv')
print('Saved: 26_shap_beeswarm.png')
print('Saved: 27_shap_bar.png')

In [ ]:
# Final inventory of all report files
import glob

files = sorted(glob.glob(f'{REPORT_DIR}/*'))
print(f'=== {len(files)} report files saved to {REPORT_DIR} ===')
print()
for f in files:
    size = os.path.getsize(f)
    ftype = 'CSV' if f.endswith('.csv') else 'PNG' if f.endswith('.png') else 'OTHER'
    print(f'  [{ftype}] {os.path.basename(f):<55} {size:>8,} bytes')